# San Francisco, Starring Parking

Every parking lot mapped in OpenStreetMap, then all of them squeezed into one square. Run `fetch_week1.py day3` first.

## 1 · Settings

In [ ]:
DAY      = 3
PROMPT   = "Polygons"
SLUG     = "polygons-sf-parking"
NOTEBOOK = "day03_polygons_sf_parking.ipynb"
TITLE    = "San Francisco,\nStarring Parking"
CREDITS  = "Parking and streets: © OpenStreetMap contributors. City boundary: US Census Bureau. Only lots mapped so far, so the real total is likely higher."

## 2 · Load

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, pandas as pd
from mapkit import poster, publish
from mapkit.paths import PROCESSED

parking = gpd.read_file(PROCESSED / "sf_parking.geojson")
streets = gpd.read_file(PROCESSED / "sf_streets.geojson")
park = gpd.read_file(PROCESSED / "sf_ggpark.geojson")
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'sf'")

## 3 · Analysis, every number on the map comes from here

In [ ]:
UTM = 32610
kinds = parking["parking"].fillna("untagged")
# Street-side strips and lanes are not lots, and underground garages don't take up the surface
lots = parking[~kinds.isin(["street_side", "lane", "underground", "on_kerb", "half_on_kerb", "shoulder"])].copy()
lots["km2"] = lots.to_crs(UTM).area / 1e6
area = lots.km2.sum()
park_km2 = park.to_crs(UTM).area.sum() / 1e6
land_km2 = city.to_crs(UTM).area.sum() / 1e6
print(f"All mapped parking   {len(parking):,} shapes, {parking.to_crs(UTM).area.sum()/1e6:.2f} km2")
print(f"Lots on the map      {len(lots):,} shapes, {area:.2f} km2, {area*247.105:.0f} acres")
print(f"Golden Gate Park     {park_km2:.2f} km2")
print(f"Lots vs the park     {area/park_km2*100:.0f}%")
print(f"Share of city land   {area/land_km2*100:.1f}%")
print(lots.parking.fillna('untagged').value_counts())

## 4 · Map

In [ ]:
HOT, STREET, LOT = "#ff2d78", "#c9135c", "#0b0b0d"
fig, ax = poster.page(HOT)
to_marin = streets.cx[-122.60:-122.45, 37.812:38.0]           # the Golden Gate Bridge runs off toward Marin
streets_m = streets.drop(index=to_marin.index).to_crs(3857)
streets_m.plot(ax=ax, color=STREET, lw=0.42, zorder=2)
lots_m = lots.to_crs(3857)
lots_m.plot(ax=ax, color=LOT, edgecolor=LOT, lw=0.5, zorder=4)
ax.set_axis_off()
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.5165, 37.704, -122.355, 37.8125)
h = fy1 - fy0
ax.set_ylim(fy0 + h * 0.03, fy1 + h * 0.03)                   # slide the city down, clear of the title
fy0, fy1 = ax.get_ylim()

# all the lots, squeezed into one square at the same scale
side_m = (area * 1e6) ** 0.5
one_m = 1 / np.cos(np.radians(37.76))                         # mercator stretches distance at this latitude
side = side_m * one_m
sx, sy = fx0 + (fx1 - fx0) * 0.06, fy0 + h * 0.058
ax.add_patch(plt.Rectangle((sx, sy), side, side, color=LOT, zorder=6))
ax.text(sx + side + (fx1 - fx0) * 0.022, sy + side * 0.5, f"All of it in one lot,\nat the same scale: {area * 247.105:.0f} acres",
        color=LOT, fontproperties=poster.font(poster.SANS_BOLD, 15), va="center", linespacing=1.3, zorder=6)

fig.text(0.06, 0.968, "San Francisco,", color=LOT, fontproperties=poster.font(poster.SERIF, 56), va="top")
fig.text(0.06, 0.909, "Starring Parking", color=LOT, fontproperties=poster.font(poster.SERIF, 56), va="top")
fig.text(0.06, 0.846, "Every parking lot mapped in the city, in black.\nTogether, they cover about half of Golden Gate Park.", color=LOT,
         fontproperties=poster.font(poster.SANS, 15.5), va="top", linespacing=1.35)
poster.credit(fig, CREDITS, color="#4d0a24")

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle="Every mapped parking lot in the city, about half of Golden Gate Park", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)